# Efficient Edge AI - MNIST hands-on with TensorFlow/Keras

**Goal:** compare a dense Multi-Layer Perceptron (MLP) with a small CNN on **MNIST** handwritten digits. Both models classify grayscale $28\times28$ images into the digits 0-9.

You will implement and train both models, then compare training/validation curves, test accuracy, parameter count and memory, MACs per image, and batch-1 inference latency.

## Architectures

**Dense MLP:** `28×28×1 -> Flatten -> Dense(128) -> Dense(64) -> Dense(10)`

**Small CNN:** `28×28×1 -> Conv3×3(8) -> MaxPool2 -> Conv3×3(16) -> MaxPool2 -> Flatten -> Dense(32) -> Dense(10)`

Use ReLU after hidden Conv/Dense layers. The last layer outputs **logits**, not probabilities. The intentionally small architectures make the comparison suitable for edge AI.

Suggested time: **45-60 min**.

## 0. Running in Google Colab

[Open Google Colab](https://colab.research.google.com/), then choose **File -> Upload notebook** and select this `.ipynb` file. After publishing it on GitHub, the direct link will be `https://colab.research.google.com/github/alessandrocapotondi/edge_ai_course/blob/main/edge_ai_mnist_tensorflow_hands_on.ipynb`.

For latency measurements, select **Runtime -> Change runtime type -> GPU**. MNIST also works on a CPU; a GPU only makes training and measurements faster.

We do not apply data augmentation; we only convert pixels to `float32` in the `[0, 1]` range.

In [ ]:
import os
import time
import random
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3
NUM_CLASSES = 10

## 1. Load MNIST

`tf.keras.datasets.mnist.load_data()` provides 60,000 training images and 10,000 test images. The original data has shape `28×28`; we explicitly add the final channel to obtain `28×28×1`, the format expected by `Conv2D`. We reserve 5,000 examples for validation.

In [ ]:
# MNIST returns 28 x 28 images and integer labels from 0 to 9.
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Scale the pixels and add the grayscale channel required by Conv2D: (N, 28, 28, 1).
x_train_full = x_train_full.astype("float32")[..., np.newaxis] / 255.0
x_test = x_test.astype("float32")[..., np.newaxis] / 255.0
y_train_full = y_train_full.astype("int64")
y_test = y_test.astype("int64")

# Deterministic split: 5,000 examples for validation and 55,000 for training.
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(x_train_full))
val_idx = indices[:5_000]
train_idx = indices[5_000:]
x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

# Create batched, prefetched pipelines for efficient input processing.
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Train:", x_train.shape, y_train.shape)
print("Val:  ", x_val.shape, y_val.shape)
print("Test: ", x_test.shape, y_test.shape)

In [ ]:
class_names = [str(digit) for digit in range(NUM_CLASSES)]

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), x_train[:10], y_train[:10]):
    # Remove the final channel to display a single grayscale image.
    ax.imshow(image.squeeze(-1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(class_names[int(label)])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Exercise - implement the MLP

Implement `Input(28, 28, 1) -> Flatten -> Dense(128, ReLU) -> Dense(64, ReLU) -> Dense(10 logits)`.

1. Return a `tf.keras.Sequential` named `dense_mlp`.
2. The first layer must be `Input(shape=(28, 28, 1))`.
3. `Flatten` transforms each image into 784 values; add the two hidden Dense layers with `activation="relu"`.
4. The final `Dense(NUM_CLASSES)` must not have an `activation`: `SparseCategoricalCrossentropy(from_logits=True)` computes softmax internally.

Do not add dropout or batch normalization: the comparison must remain controlled.

In [ ]:
def build_mlp():
    # Replace the TODO and exception with this model:
    # return tf.keras.Sequential(
    #     [
    #         tf.keras.layers.Input(shape=(28, 28, 1)),  # grayscale MNIST input
    #         tf.keras.layers.Flatten(),                 # 28 x 28 x 1 -> 784
    #         tf.keras.layers.Dense(128, activation="relu"),
    #         tf.keras.layers.Dense(64, activation="relu"),
    #         tf.keras.layers.Dense(NUM_CLASSES),         # logits: no softmax here
    #     ],
    #     name="dense_mlp",
    # )
    raise NotImplementedError("Implement build_mlp()")

# mlp = build_mlp()
# mlp.summary()

### Self-check for the MLP

Run this cell after implementing the model. It checks the input/output shape and the expected parameter count.

In [ ]:
def check_mlp(model):
    # Input and output shape expose missing channels or a wrong final layer.
    assert model.input_shape == (None, 28, 28, 1), model.input_shape
    assert model.output_shape == (None, NUM_CLASSES), model.output_shape
    assert model.count_params() == 109_386, (
        f"Unexpected parameter count: {model.count_params():,}"
    )
    print("MLP architecture looks correct.")
    print(f"Parameters: {model.count_params():,}")

# check_mlp(mlp)

## 3. Exercise - implement the small CNN

Implement `Input(28, 28, 1) -> Conv3×3(8, same, ReLU) -> MaxPool2×2 -> Conv3×3(16, same, ReLU) -> MaxPool2×2 -> Flatten -> Dense(32, ReLU) -> Dense(10 logits)`.

1. Configure the convolutions with `strides=1` and `padding="same"`.
2. Each `MaxPool2D(pool_size=2, strides=2)` halves the spatial dimensions: $28	o14	o7$.
3. After the second pooling layer, `Flatten` must produce $7×7×16 = 784$ features.
4. Finish with `Dense(32, activation="relu")` and `Dense(NUM_CLASSES)` without an activation.

In [ ]:
def build_cnn():
    # Replace the TODO and exception with a Sequential model containing:
    # Input(shape=(28, 28, 1)),
    # Conv2D(8, 3, strides=1, padding="same", activation="relu"),
    # MaxPool2D(pool_size=2, strides=2),  # 28 x 28 -> 14 x 14
    # Conv2D(16, 3, strides=1, padding="same", activation="relu"),
    # MaxPool2D(pool_size=2, strides=2),  # 14 x 14 -> 7 x 7
    # Flatten(),                           # 7 x 7 x 16 -> 784
    # Dense(32, activation="relu"),
    # Dense(NUM_CLASSES),                  # logits, no softmax
    # Use `name="small_cnn"` for the model.
    raise NotImplementedError("Implement build_cnn()")

# cnn = build_cnn()
# cnn.summary()

### Self-check for the CNN

In [ ]:
def check_cnn(model):
    assert model.input_shape == (None, 28, 28, 1), model.input_shape
    assert model.output_shape == (None, NUM_CLASSES), model.output_shape
    assert model.count_params() == 26_698, (
        f"Unexpected parameter count: {model.count_params():,}"
    )

    # Probe the Flatten output to confirm that pooling produced 7 x 7 x 16.
    flatten_layers = [layer for layer in model.layers if isinstance(layer, tf.keras.layers.Flatten)]
    assert len(flatten_layers) == 1, "Expected exactly one Flatten layer."
    probe = tf.keras.Model(model.input, flatten_layers[0].output)
    features = probe(tf.zeros((1, 28, 28, 1)))
    assert features.shape[-1] == 784, f"Expected 784 features, got {features.shape[-1]}"

    print("CNN architecture looks correct.")
    print(f"Parameters: {model.count_params():,}")
    print("Flatten features:", int(features.shape[-1]))

# check_cnn(cnn)

## 4. Compile and train both models

To keep the comparison controlled, use for both models:

- Adam, learning rate `1e-3`;
- sparse categorical cross-entropy **from logits**;
- the same batch size and number of epochs.

**Exercise:** instantiate both models, compile them, and train them.

In [ ]:
def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )
    return model

# TODO:
# 1) instantiate mlp and cnn
# 2) run check_mlp() and check_cnn()
# 3) compile both models
# 4) train each model using train_ds and validation_data=val_ds
#
# Expected variables at the end:
#   mlp, cnn, history_mlp, history_cnn
#
# Example skeleton:
#
# mlp = compile_model(build_mlp())
# cnn = compile_model(build_cnn())
# check_mlp(mlp)
# check_cnn(cnn)
#
# history_mlp = mlp.fit(...)
# history_cnn = cnn.fit(...)

## 5. Compare training curves

Plot training and validation loss/accuracy for the two models. Look for:

- convergence speed;
- final validation accuracy;
- train/validation gap;
- signs of underfitting or overfitting.

In [ ]:
def plot_histories(history_mlp, history_cnn):
    epochs_mlp = range(1, len(history_mlp.history["loss"]) + 1)
    epochs_cnn = range(1, len(history_cnn.history["loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_mlp, history_mlp.history["loss"], label="MLP train")
    plt.plot(epochs_mlp, history_mlp.history["val_loss"], label="MLP val")
    plt.plot(epochs_cnn, history_cnn.history["loss"], label="CNN train")
    plt.plot(epochs_cnn, history_cnn.history["val_loss"], label="CNN val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training curves — loss")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_mlp, history_mlp.history["accuracy"], label="MLP train")
    plt.plot(epochs_mlp, history_mlp.history["val_accuracy"], label="MLP val")
    plt.plot(epochs_cnn, history_cnn.history["accuracy"], label="CNN train")
    plt.plot(epochs_cnn, history_cnn.history["val_accuracy"], label="CNN val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training curves — accuracy")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

# plot_histories(history_mlp, history_cnn)

## 6. Test accuracy

In [ ]:
def test_accuracy(model):
    loss, acc = model.evaluate(test_ds, verbose=0)
    return float(loss), float(acc)

# mlp_test_loss, mlp_test_acc = test_accuracy(mlp)
# cnn_test_loss, cnn_test_acc = test_accuracy(cnn)
# print(f"MLP test accuracy: {mlp_test_acc:.4f}")
# print(f"CNN test accuracy: {cnn_test_acc:.4f}")

## 7. Model size in memory and on disk

We report two related but different quantities:

1. **Parameter footprint** = number of stored parameter bytes. With float32, this is approximately `4 × #parameters`.
2. **Serialized weight-file size** = actual `.weights.h5` file size, which also includes file-format metadata.

This is **not** the full runtime memory footprint: activations, framework buffers, allocator overhead, and temporary workspaces are not included.

In [ ]:
def parameter_memory_bytes(model):
    total = 0
    for v in model.weights:
        # tf.DType.size gives bytes per scalar.
        total += int(np.prod(v.shape)) * int(v.dtype.size)
    return total

def saved_weights_size_bytes(model, filename):
    path = Path(tempfile.gettempdir()) / filename
    model.save_weights(path)
    return path.stat().st_size

# Example:
# mlp_param_bytes = parameter_memory_bytes(mlp)
# cnn_param_bytes = parameter_memory_bytes(cnn)
# mlp_file_bytes = saved_weights_size_bytes(mlp, "mlp.weights.h5")
# cnn_file_bytes = saved_weights_size_bytes(cnn, "cnn.weights.h5")
#
# print("MLP parameter footprint [MiB]:", mlp_param_bytes / 2**20)
# print("CNN parameter footprint [MiB]:", cnn_param_bytes / 2**20)

## 8. Count MACs per image

We count MACs for **Conv2D** and **Dense** layers only.

- Conv2D: `Hout × Wout × Cout × Kh × Kw × Cin`
- Dense: `Nin × Nout`

Bias additions, ReLU, pooling, and data movement are excluded.

This matches the convention used in the lesson.

In [ ]:
def count_macs_keras(model, input_shape=(1, 28, 28, 1), verbose=True):
    shape = tuple(input_shape)
    total_macs = 0
    rows = []

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.InputLayer):
            continue

        out_shape = tuple(layer.compute_output_shape(shape))
        macs = 0
        if isinstance(layer, tf.keras.layers.Conv2D):
            _, h_out, w_out, c_out = out_shape
            k_h, k_w = layer.kernel_size
            macs = int(h_out * w_out * c_out * k_h * k_w * shape[-1])
        elif isinstance(layer, tf.keras.layers.Dense):
            macs = int(shape[-1] * layer.units)

        rows.append((layer.name, layer.__class__.__name__, shape, out_shape, macs))
        total_macs += macs
        shape = out_shape

    if verbose:
        df = pd.DataFrame(rows, columns=["layer", "type", "input_shape", "output_shape", "MACs"])
        display(df)
        print(f"Total MACs / image: {total_macs:,}")
    return total_macs

# mlp_macs = count_macs_keras(mlp)
# cnn_macs = count_macs_keras(cnn)

### Manual MAC check

Calculate the MACs before running `count_macs_keras()`. One MAC is a multiplication plus an accumulation; bias, ReLU, and pooling are not included.

**CNN**

- Conv1: `28 x 28 x 8 x 3 x 3 x 1 = ?`
- Conv2: `14 x 14 x 16 x 3 x 3 x 8 = ?`
- Dense32: `784 x 32 = ?`
- Dense10: `32 x 10 = ?`
- **Total = ?**

**MLP**

- Dense128: `784 x 128 = ?`
- Dense64: `128 x 64 = ?`
- Dense10: `64 x 10 = ?`
- **Total = ?**

## 9. Batch-1 inference latency

We benchmark a single image, after warm-up.

**Important:** latency depends on hardware, TensorFlow version, kernel selection, graph compilation, and synchronization. Use it as an empirical measurement, not as a hardware-independent property of the network.

Compare the two TensorFlow models on the **same Colab runtime**. Do not interpret a TensorFlow-vs-PyTorch latency difference as a pure architecture difference.

In [ ]:
def benchmark_latency_ms(model, x_sample, warmup=30, runs=200):
    x_sample = tf.convert_to_tensor(x_sample, dtype=tf.float32)

    @tf.function
    def infer(x):
        return model(x, training=False)

    # Trigger tracing + warm-up.
    for _ in range(warmup):
        y = infer(x_sample)
        _ = y.numpy()  # force device synchronization

    times_ms = []
    for _ in range(runs):
        t0 = time.perf_counter()
        y = infer(x_sample)
        _ = y.numpy()  # force device synchronization
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1e3)

    times_ms = np.asarray(times_ms)
    return {
        "mean_ms": float(times_ms.mean()),
        "median_ms": float(np.median(times_ms)),
        "p95_ms": float(np.percentile(times_ms, 95)),
    }

# x_one = x_test[:1]
# mlp_latency = benchmark_latency_ms(mlp, x_one)
# cnn_latency = benchmark_latency_ms(cnn, x_one)
# print("MLP:", mlp_latency)
# print("CNN:", cnn_latency)

## 10. Final comparison table

Run this cell after all previous measurements are available.

In [ ]:
def mib(n_bytes):
    return n_bytes / (2**20)

# TODO: uncomment after training + measurements.
#
# results = pd.DataFrame([
#     {
#         "model": "MLP",
#         "test_accuracy": mlp_test_acc,
#         "parameters": mlp.count_params(),
#         "parameter_MiB": mib(mlp_param_bytes),
#         "weight_file_MiB": mib(mlp_file_bytes),
#         "MACs_per_image": mlp_macs,
#         "latency_median_ms": mlp_latency["median_ms"],
#         "latency_p95_ms": mlp_latency["p95_ms"],
#     },
#     {
#         "model": "CNN",
#         "test_accuracy": cnn_test_acc,
#         "parameters": cnn.count_params(),
#         "parameter_MiB": mib(cnn_param_bytes),
#         "weight_file_MiB": mib(cnn_file_bytes),
#         "MACs_per_image": cnn_macs,
#         "latency_median_ms": cnn_latency["median_ms"],
#         "latency_p95_ms": cnn_latency["p95_ms"],
#     },
# ]).set_index("model")
#
# display(results.style.format({
#     "test_accuracy": "{:.4f}",
#     "parameter_MiB": "{:.3f}",
#     "weight_file_MiB": "{:.3f}",
#     "MACs_per_image": "{:,.0f}",
#     "latency_median_ms": "{:.3f}",
#     "latency_p95_ms": "{:.3f}",
# }))

## 11. Discussion questions

Write short answers based on **your measurements**, not on intuition alone.

1. The MLP and CNN have a similar MAC count. Why can their test accuracy still differ substantially?
2. Which model has more parameters? Where are most of those parameters located?
3. Does lower parameter count automatically imply lower latency? Explain using your measurement.
4. Why can two networks with similar MAC counts have different latency on the same GPU?
5. Why is the serialized file size not exactly equal to `4 × #parameters`?
6. If all weights were quantized from FP32 to INT8, what would you expect to happen to the **weight memory footprint**?
7. Which metrics here are properties of the **model**, and which depend strongly on the **runtime/hardware**?

## 12. Optional extension

Repeat the latency benchmark with batch sizes `1`, `8`, `32`, and `128`.

Plot:

- latency per batch;
- latency per image;
- throughput in images/s.

This separates **interactive latency** from **throughput**, an important distinction for edge deployment.

In [ ]:
# OPTIONAL EXERCISE
#
# Write a function that benchmarks several batch sizes and returns
# a DataFrame with:
#   batch_size, batch_latency_ms, ms_per_image, images_per_second
#
# Then compare MLP vs CNN.

---
### Checks and references

These values do not depend on training:

- MLP: **109,386 parameters**, **109,184 MAC/image**;
- CNN: **26,698 parameters**, **307,648 MAC/image**.

Accuracy and latency are empirical: report the values obtained from your run.

### Official documentation

- [Keras: MNIST classification](https://keras.io/examples/vision/mnist_convnet/)
- [Keras Sequential](https://keras.io/api/models/sequential/)
- [Keras Conv2D](https://keras.io/api/layers/convolution_layers/convolution2d/)
- [Google Colab guide](https://colab.research.google.com/notebooks/intro.ipynb)

## 13. Optional task - extend to CIFAR-10

After completing MNIST, try CIFAR-10. The dataset uses RGB `32×32×3` images: update the input, visualization, and layer shapes. Keep the loss, metrics, and number of epochs unchanged to compare architecture cost rather than a different setup. The following cell provides a starting point without changing the main MNIST workflow.

In [ ]:
# OPTIONAL: CIFAR-10 extension
# 1) Load CIFAR-10 and rescale the RGB pixel values:
# # (cifar_x_train, cifar_y_train), (cifar_x_test, cifar_y_test) = tf.keras.datasets.cifar10.load_data()
# # cifar_x_train = cifar_x_train.astype("float32") / 255.0
#
# 2) Copy build_mlp() and build_cnn(), then change:
# # Input(shape=(32, 32, 3))
# # First MLP Dense still follows Flatten, whose input is now 32 * 32 * 3.
# # CNN filters: 8 then 16; after pooling 32 -> 16 -> 8, Flatten has 16 * 8 * 8 features.
#
# 3) Compile with SparseCategoricalCrossentropy(from_logits=True), train on
# #    CIFAR-10, and run count_macs_keras(model, input_shape=(1, 32, 32, 3)).
# # Compare the result with the main MNIST experiment.